In [110]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

In [ ]:
# I tried a few different multilayer models, and they all underperformed compared to this one.
class SingleLayer(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, 8),
            nn.ReLU(),
            #nn.Dropout(0.4), # works slightly better without the dropout layer, and considering we're already doing mini-batching and a train-test split, I'm not too concerned about overfitting
            nn.Linear(8, 1))
    def forward(self, x):
        return(self.sequential(x))

In [ ]:
mystery_data = pd.read_csv("FP_Data.csv")

mystery_data_onehot = pd.get_dummies(mystery_data)

y = mystery_data_onehot.pop("y")
X = mystery_data_onehot

X = normalize(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.25, random_state=28)

In [ ]:
# How well does a linear model do, in terms of MSE, when trained on the train data?

from sklearn.linear_model import LinearRegression

baseline_lm = LinearRegression().fit(X_train, y_train)
lm_mse = np.mean((baseline_lm.predict(X_test) - y_test)**2)
print(f"Baseline Linear Model Validation MSE: {lm_mse:.4f}")

Baseline Linear Model Validation MSE: 125.3744


In [ ]:
model = SingleLayer(11)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1) # This is, as far as I seen, the most widely used optimizer, though it is not what is used in the textbook
loss_fn = nn.MSELoss()

epochs = 50 # The model appears to stabilize by the 50th epoch
batch_size = 32

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.float32)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.float32)

for epoch in range(epochs):
    model.train()

    permutation = torch.randperm(X_train.size(0))
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        X_batch, y_batch = X_train[indices], y_train[indices]

        optimizer.zero_grad()
        output = model(X_batch).squeeze()
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(X_test).squeeze()
        val_loss = loss_fn(val_output, y_test)
    print(f"Epoch {epoch+1}/{epochs} | Val Loss: {val_loss.item():.4f}")

Epoch 1/50 | Val Loss: 3175.7400
Epoch 2/50 | Val Loss: 3108.2173
Epoch 3/50 | Val Loss: 3011.2493
Epoch 4/50 | Val Loss: 2883.0645
Epoch 5/50 | Val Loss: 2722.3862
Epoch 6/50 | Val Loss: 2529.2163
Epoch 7/50 | Val Loss: 2306.8542
Epoch 8/50 | Val Loss: 2058.1077
Epoch 9/50 | Val Loss: 1787.3077
Epoch 10/50 | Val Loss: 1503.3394
Epoch 11/50 | Val Loss: 1215.8517
Epoch 12/50 | Val Loss: 938.4147
Epoch 13/50 | Val Loss: 689.9064
Epoch 14/50 | Val Loss: 488.7353
Epoch 15/50 | Val Loss: 350.2524
Epoch 16/50 | Val Loss: 278.0042
Epoch 17/50 | Val Loss: 256.8469
Epoch 18/50 | Val Loss: 254.7601
Epoch 19/50 | Val Loss: 246.8639
Epoch 20/50 | Val Loss: 223.5552
Epoch 21/50 | Val Loss: 194.3138
Epoch 22/50 | Val Loss: 172.0848
Epoch 23/50 | Val Loss: 161.1849
Epoch 24/50 | Val Loss: 160.0964
Epoch 25/50 | Val Loss: 163.7640
Epoch 26/50 | Val Loss: 168.6967
Epoch 27/50 | Val Loss: 171.6748
Epoch 28/50 | Val Loss: 169.9242
Epoch 29/50 | Val Loss: 164.2142
Epoch 30/50 | Val Loss: 155.9169
Epoch 31

In [ ]:
# This is also just the last validation loss

model.eval()
with torch.no_grad():
    y_pred = model(X_test).squeeze()
    mse = loss_fn(y_pred, y_test)
    print(f"Test MSE: {mse.item():.4f}")

Test MSE: 126.0813


While the nueral network underperforms the baseline linear model run on the full dataset, if we look at the baseline linear model run only on the train data, then tested on the test data, they are comparable.